In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
import scipy.optimize
import datetime
import sys

class gel_3D:
    def __init__(self, length=90.0, width =15.0, thickness=1.6, phi0=0.2, mu_hat=+0.01 * 136.6):
        self.phi0 = phi0
        self.mu_hat = mu_hat   # 🔥 NUEVO
        
        self.entropic_unit = 136.6  # measured in MPa
        self.G = 0.13               # measured in MPa
        self.gamma = self.G/self.entropic_unit
        self.chi =  0.348
        self.density =  1.23 # measured in [g/mL]

        self.L = length      # measured in mm
        self.d = thickness    # measured in mm
        self.w = width # measured in mm

        self.filename_suffix = "_phi0={:.1f}_mesh0Polymer".format(self.phi0)

        def auxIsotropic(s):
            return s*self.dH(s*s*s) + self.gamma
        self.lambda_iso = scipy.optimize.fsolve(auxIsotropic, 1.7)[0]

        def auxUniaxial(s):
            return s*self.gamma + self.dH(s)
        self.lambda_target = scipy.optimize.fsolve(auxUniaxial, 1.9)[0]
        
        def auxEnergyDensity(lambda1, lambda2, lambda3):
            gel=self; phi0=gel.phi0; G=gel.G; chi=gel.chi; nu=gel.entropic_unit
            J= lambda1*lambda2*lambda3
            phi = phi0/J
            return 0.5*G*(lambda1**2 + lambda2**2 + lambda3**2) + nu*((J-phi0)*np.log(1-phi) + phi0*chi*(1-phi))

        lambda_iso = self.lambda_iso
        self.reference_energy_density = auxEnergyDensity(lambda_iso, lambda_iso, lambda_iso)                

    def phi(self, J):
        return self.phi0/J

    def H(self, J):
        return (J - self.phi0)*log(1-self.phi(J))  + self.phi0 * self.chi*(1-self.phi(J))

    def dH(self, J):
        return self.phi(J) + np.log(1-self.phi(J)) + self.chi * self.phi(J)**2

    def Gfun(self, lamb):
        nu = self.entropic_unit
        return (-self.dH(lamb)/lamb)*nu
        
    # energy density in [MPa]
    def W(self, F):
               
        J = Det(F)
        C_tensor = F.trans * F
        
        gel = self
        G = gel.G
        nu = gel.entropic_unit
        phi0 = gel.phi0
        mu_hat = gel.mu_hat
        
        reference_energy_density =  self.reference_energy_density
        
        # 🔥 concentración
        C = (J - phi0) / nu
        
        return 0.5*G*(Trace(C_tensor)) + nu*gel.H(J) - mu_hat*C - reference_energy_density

In [2]:
class Solve_gel3d:
    def __init__(self, gel, order=1):         # corner_refinement capability not yet activated
        self.gel = gel
        self.order = order
        self.start_time = datetime.datetime.now()  

    def add_mesh(self, mesh_file):        
        self.mesh = Mesh(mesh_file)
    
    def Space(self):
        # Finite element space with zero-displacement boundary condition 
        # on all of the bottom interface
        self.fes = VectorH1(self.mesh, order=self.order, dirichlet="bonded|debonded")
        print('nDoF = {}'.format(self.fes.ndof))
        
    def model(self):
        u  = self.fes.TrialFunction()
        I = Id(self.mesh.dim)
        F = I + Grad(u)

        gravity = CoefficientFunction((0,0,-9.8))
        gel_density_in_g_per_mL = self.gel.density

        def negpart(var):
            return (sqrt(var**2)-var)*0.5        
        
        AA = 1e5

        # hydrogel model        
        self.a = BilinearForm(self.fes, symmetric=False)

        # 🔥 AQUÍ ya entra μ automáticamente desde gel.W(F)
        self.a += Variation(self.gel.W(F).Compile() * dx)

        # gravedad
        self.a += Variation( -((1e-6)*gel_density_in_g_per_mL)*InnerProduct(gravity, u)*dx )

        # contacto
        self.a += Variation(AA*negpart(y+u[1])**2 * dx)
        
    def Solve_incremental_softening(self):
        self.Space()
        self.gfu = GridFunction(self.fes)
        self.gfu.vec[:] = 0
        
        lambda_initial = 1.1
        G_end = self.gel.G
        G0 = self.gel.Gfun(lambda_initial)

        nIterations = 15
        self.gel.G = Parameter(G0)

        self.model()
        
        lambda_list = np.linspace(lambda_initial, self.gel.lambda_target, nIterations, endpoint=False)
        G_list = [self.gel.Gfun(la) for la in lambda_list]
        
        G_list.append(G_end)
              
        filename='gridfunctions/result'+ \
            self.gel.filename_suffix+ \
            "_order={}".format(self.order)

        tol=1e-3
        maxits=100
        
        #indexes_iterations = range(nIterations+1)
        indexes_iterations = [0]

        for numIteration in indexes_iterations:
            G_i = G_list[numIteration]
            print("*** Iteration #", numIteration, ". Shear modulus G = ", G_i)

            if numIteration == nIterations:
                tol = 1e-6
                maxits = 500
            
            self.gel.G.Set(G_i)

            self.gfu, _, _ = SolveNonlinearMinProblem(
                a=self.a,
                gfu=self.gfu,
                maxits=maxits,
                tol=tol,
                alpha=1e-2
            )

            self.gfu.Save(filename + '_iter=' + str(numIteration).zfill(2) + '.gfu')

            print("Total time elapsed =" + str(datetime.datetime.now() - self.start_time))

In [3]:
# experimental acceleration
def SolveNonlinearMinProblem(a, gfu, tol=1e-08, maxits=50, alpha=5e-2):
    
    start_time = datetime.datetime.now()  

    res = gfu.vec.CreateVector()
    du  = gfu.vec.CreateVector()
    
    # precond local, multigrid, bddc
    precond = 'bddc'
    c = Preconditioner(a, precond)

    for it in range(maxits):

        with TaskManager():
            a.Apply(gfu.vec, res)
            c = Preconditioner(a, precond)
            a.AssembleLinearization(gfu.vec)
            
            c.Update()
            inv = CGSolver(a.mat, c.mat, maxsteps=1000)

            du.data = alpha * inv * res

            # update
            gfu.vec.data -= du

        # stopping criteria
        stopcritval = sqrt(abs(InnerProduct(du, res)))

        print("Newton iteration:", it, "Time elapsed =" + str(datetime.datetime.now() - start_time))
        print("<A u", it, ", A u", it, ">_{-1}^0.5 = ", stopcritval)

        if stopcritval < tol:
            break

    return gfu, stopcritval, it

In [4]:
### MAIN ###

# data = [dummy, L, w, d, phi0]
data = ['','90', '15.0', '1.62', '0.2'] 
order = 1

mesh_file = 'meshes/mesh0.vol.gz'

L = float(data[1])
d = float(data[2])
w = float(data[3])
phi0 = float(data[4])

print(f'L={L}, w={w}, d={d}, phi0={phi0}')

# 🔥 DEFINE EL POTENCIAL QUÍMICO
mu_hat = -0.05 * 136.6   # ejemplo (en MPa, consistente con tu modelo)

gel = gel_3D(length=L, width=w, thickness=d, phi0=phi0, mu_hat=mu_hat)

modelling = Solve_gel3d(gel, order=order)
modelling.add_mesh(mesh_file)
modelling.Space()

L=90.0, w=1.62, d=15.0, phi0=0.2
nDoF = 90978


In [5]:
modelling = Solve_gel3d (gel, order=order)
modelling.add_mesh(mesh_file)
modelling.Solve_incremental_softening()

nDoF = 90978
*** Iteration # 0 . Shear modulus G =  0.9125353645057322
Newton iteration: 0 Time elapsed =0:00:01.416909
<A u 0 , A u 0 >_{-1}^0.5 =  0.7377479368254537
Newton iteration: 1 Time elapsed =0:00:02.652310
<A u 1 , A u 1 >_{-1}^0.5 =  0.73095627589445
Newton iteration: 2 Time elapsed =0:00:03.747719
<A u 2 , A u 2 >_{-1}^0.5 =  0.7242246783104642
Newton iteration: 3 Time elapsed =0:00:04.931347
<A u 3 , A u 3 >_{-1}^0.5 =  0.7175525262107987
Newton iteration: 4 Time elapsed =0:00:06.111847
<A u 4 , A u 4 >_{-1}^0.5 =  0.7109392176659265
Newton iteration: 5 Time elapsed =0:00:07.419417
<A u 5 , A u 5 >_{-1}^0.5 =  0.7043841657220122
Newton iteration: 6 Time elapsed =0:00:08.793791
<A u 6 , A u 6 >_{-1}^0.5 =  0.6978867975135411
Newton iteration: 7 Time elapsed =0:00:09.948353
<A u 7 , A u 7 >_{-1}^0.5 =  0.6914465534404853
Newton iteration: 8 Time elapsed =0:00:11.206096
<A u 8 , A u 8 >_{-1}^0.5 =  0.6850628864048867
Newton iteration: 9 Time elapsed =0:00:12.362144
<A u 9 , 